# Sesi 11 — RAG bagian 3: Merangkai Semuanya


**Alur proses RAG kita**

```
ingest  ->  embedding  ->  indexing  ->  retrieval  ->  re-ranking
        ->  context injection  ->  generate
```

In [1]:
import os
import pathlib
import re

import faiss
import numpy as np
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

MODEL_EMBED = "nomic-embed-text:latest"
MODEL_JAWAB = "gpt-oss:20b"
MODEL_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
DIMENSI = 768
UKURAN_CHUNK = 600


def akar_sesi():
    # Folder yang memuat kb_lindungiku, entah folder ini atau folder induknya.
    for p in (pathlib.Path("."), pathlib.Path("..")):
        if (p / "kb_lindungiku").is_dir():
            return p.resolve()
    raise SystemExit(
        "Folder kb_lindungiku tidak ditemukan. Jalankan notebook ini dari folder "
        "Session 11 atau dari salah satu subfoldernya."
    )


FOLDER_KB = akar_sesi() / "kb_lindungiku"

client = ollama.Client()
reranker = CrossEncoder(MODEL_RERANK)
print("knowledge base :", FOLDER_KB)
print("siap")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

knowledge base : /Users/fbrynmghni/Code/AI Engineering/Rework Academy/Sesi 11/kb_lindungiku
siap


## Bagian 1 — Membangun index

In [2]:
def bersihkan(teks):
    teks = teks.replace("\u00a0", " ")
    teks = re.sub(r"[ \t]+", " ", teks)
    return re.sub(r"\n{3,}", "\n\n", teks).strip()


def normalisasi(v):
    panjang = np.linalg.norm(v, axis=1, keepdims=True)
    panjang[panjang == 0] = 1e-12
    return v / panjang


def embed(daftar_teks, task_type=None, batas=100):
    hasil = []
    for i in range(0, len(daftar_teks), batas):
        bagian = daftar_teks[i : i + batas]
        resp = client.embed(
            model=MODEL_EMBED,
            input=bagian,
            # TODO 4: lengkapi config-nya.
            # Pakai types.EmbedContentConfig dengan task_type=task_type
            # dan output_dimensionality=DIMENSI.
        )
        hasil.extend(resp.embeddings)
    embeddings = np.array(hasil, dtype="float32")

    if embeddings.shape[1] != DIMENSI:
        raise ValueError(
            f"Dimensi embedding {embeddings.shape[1]} tidak sesuai dengan "
            f"yang diharapkan {DIMENSI}."
        )
    return embeddings


pemisah = RecursiveCharacterTextSplitter(
    chunk_size=UKURAN_CHUNK,
    chunk_overlap=80,
    separators=["\n## ", "\n\n", "\n", ". ", " ", ""],
)

potongan = []
for berkas in sorted(FOLDER_KB.glob("*.md")):
    for teks in pemisah.split_text(bersihkan(berkas.read_text(encoding="utf-8"))):
        potongan.append({"sumber": berkas.name, "teks": teks})

index = faiss.IndexFlatIP(DIMENSI)
index.add(normalisasi(embed([p["teks"] for p in potongan], "RETRIEVAL_DOCUMENT")))

print(f"{len(potongan)} chunk, {index.ntotal} vector. index siap.")

54 chunk, 54 vector. index siap.


## Bagian 2 — Context injection

Di sinilah retrieval berubah menjadi jawaban, dan di sinilah sebagian besar sistem
RAG gagal secara diam-diam.

Kalau Anda hanya menempelkan potongan ke dalam prompt tanpa instruksi, model akan
menganggapnya sebagai bahan tambahan, bukan sebagai satu-satunya sumber. Ketika
konteksnya tidak memuat jawaban, model kembali memakai data latihnya dan
mengarang, dengan nada yang sama meyakinkannya.

Prompt di bawah melakukan tiga hal, dan ketiganya diperlukan.

1. **Membatasi sumber.** Kata HANYA ditulis dengan huruf besar karena memang
   sepenting itu.
2. **Memberi jalan keluar.** Model diberi izin eksplisit untuk mengaku tidak tahu.
   Tanpa kalimat ini, model merasa wajib menjawab apa pun.
3. **Memisahkan bagian dengan jelas.** Konteks dan pertanyaan dipisahkan oleh
   penanda, supaya model tidak salah menebak mana yang mana.

Ini adalah structured prompt yang sama seperti yang Anda pelajari sebelumnya,
dengan konteks hasil retrieval di dalamnya.

In [7]:
# TODO 1: tulis template prompt-nya.
# Harus memuat tiga hal: batasi model hanya pada konteks, beri izin untuk
# mengaku tidak tahu, dan pisahkan KONTEKS dari PERTANYAAN dengan penanda jelas.
# Sediakan dua placeholder, yaitu {konteks} dan {pertanyaan}.
TEMPLATE = """
Jawab pertanyaan HANYA berdasarkan konteks di bawah ini. 
Apabila informasi tidak ditemukan dalam konteks, 
katakan bahwa kamu tidak memiliki informasi tersebut. Jangan menebak, Jawab dengan bahasa Sunda

KONTEKS:
{konteks}

PERTANYAAN: {pertanyaan}
"""


def susun_prompt(pertanyaan, potongan_terpilih):
    # TODO 2: gabungkan potongan terpilih menjadi satu string konteks.
    # Sertakan nama sumber tiap potongan, dan pisahkan antar potongan dengan
    # pembatas yang jelas. Lalu isikan ke dalam TEMPLATE.
    konteks = "\n\n---\n\n".join(
            f"[{p['sumber']}]\n{p['teks'].strip()}" for p in potongan_terpilih
        )
    return TEMPLATE.format(konteks=konteks, pertanyaan=pertanyaan)
    
    
print(susun_prompt("Bagaimana cara klaim rawat inap", potongan[:3]))


Jawab pertanyaan HANYA berdasarkan konteks di bawah ini. 
Apabila informasi tidak ditemukan dalam konteks, 
katakan bahwa kamu tidak memiliki informasi tersebut. Jangan menebak, Jawab dengan bahasa Sunda

KONTEKS:
[01_kebijakan_klaim_kesehatan.md]
# Kebijakan Klaim Asuransi Kesehatan

Dokumen internal LindungiKu. Berlaku sejak 1 Januari 2026\. Menggantikan seluruh ketentuan klaim kesehatan sebelumnya.

---

[01_kebijakan_klaim_kesehatan.md]
## Masa Tunggu

Setiap polis kesehatan LindungiKu memiliki masa tunggu sebelum klaim dapat diajukan. Masa tunggu dihitung sejak tanggal polis dinyatakan aktif, bukan sejak tanggal pengajuan.

Masa tunggu klaim rawat inap adalah 45 hari kalender. Masa tunggu klaim rawat jalan adalah 30 hari kalender. Masa tunggu untuk penyakit kritis yang tercantum pada Lampiran A polis adalah 9 bulan.

Pengecualian berlaku untuk kecelakaan. Klaim akibat kecelakaan dapat diajukan sejak hari pertama polis aktif, tanpa masa tunggu, dengan syarat kejadian dilaporkan pa

## Bagian 3 — Fungsi `jawab`

Seluruh pipeline dalam satu fungsi. Baca dari atas ke bawah, dan Anda akan melihat
ketujuh langkahnya berurutan.

Perhatikan pola dua tahapnya. `k_ambil=8` mengambil delapan kandidat dengan
pencarian yang murah, lalu `k_pakai=3` menyisakan tiga terbaik setelah diurutkan
ulang oleh cross-encoder. Mengambil lebih banyak dari yang dipakai adalah
intinya, karena re-ranker hanya bisa memperbaiki urutan dari kandidat yang
diberikan kepadanya.

In [9]:
def jawab(pertanyaan, k_ambil=8, k_pakai=3, tampilkan=True):
    # 1. pertanyaan menjadi vector
    q = normalisasi(embed([pertanyaan], "RETRIEVAL_QUERY"))

    # TODO 3: cari k_ambil kandidat terdekat di dalam index,
    # lalu ambil potongan yang sesuai dengan nomor baris hasil pencarian.
    _, baris = index.search(q, k_ambil)
    kandidat = [
        potongan[i] for i in baris[0]
        if i != -1
        ]

    # TODO 4: nilai setiap pasangan (pertanyaan, teks potongan) dengan reranker,
    # urutkan dari nilai tertinggi, lalu sisakan k_pakai teratas.
    nilai = reranker.predict(
        [(pertanyaan, p["teks"]) for p in kandidat],
        batch_size=1,
        show_progress_bar=False
    )
    
    terpilih = [
        p for _, p in sorted(zip(nilai, kandidat), 
                             key=lambda x: -x[0])[:k_pakai]
                             ]

    prompt = susun_prompt(pertanyaan, terpilih)

    # TODO 5: kirim prompt ke model memakai client.models.generate_content
    # dengan model=MODEL_JAWAB, lalu simpan hasilnya ke resp.
    resp = client.chat(
        model=MODEL_JAWAB,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.1
        }
    )

    jawaban = resp.message.content.strip()

    if tampilkan:
        print("SUMBER YANG DIPAKAI")
        for p in terpilih:
            print("   -", p["sumber"])
        print()
        print("JAWABAN")
        print(jawaban)

    return jawaban, terpilih


_ = jawab("Bagaimana cara klaim rawat inap?")

SUMBER YANG DIPAKAI
   - 01_kebijakan_klaim_kesehatan.md
   - 07_faq_internal_agen.md
   - 01_kebijakan_klaim_kesehatan.md

JAWABAN
**Kumaha cara klaim rawat inap?**

1. **Kumpulkeun dokumén nu diperlukeun**  
   - Rincian biaya asli ti rumah sakit (bukti pembayaran, struk, jeung dokumén resmi).  
   - Dokumén identitas diri (KTP, kartu asuransi, jsb).  
   - Formulir klaim (bisa diisi ku agen atawa langsung diunggah ka sistem).

2. **Pastikeun wanci pangajuan**  
   - Klaim kudu diajukeun **paling lami 60 dinten kalénder** ti tanggal kaluar ti rumah sakit.  
   - Lamun kirang ti 60 dinten, klaim bakal ditolak sacara otomatis ku sistem.

3. **Unggah dokumén ka sistem**  
   - Unggah dokumén asli (henteu foto layar aplikasi rumah sakit).  
   - Pastikeun unggahan jelas jeung lengkep.

4. **Verifikasi**  
   - Sistem bakal ngalakukeun verifikasi **2 dinten kerja** sanggeus dokumén lengkep.  
   - Dina tahap ieu, pihak asuransi mariksa kecocokan jeung syarat klaim.

5. **Co‑payment (lamun

## Bagian 4 — Uji kejujuran

Sistem yang menjawab benar ketika jawabannya ada di dokumen itu bagus. Sistem yang
mengaku tidak tahu ketika jawabannya memang tidak ada di dokumen itu **jauh lebih
bagus**, dan jauh lebih jarang.

Pertanyaan di bawah tidak dibahas sama sekali oleh ketujuh dokumen kita.

Kalau model tetap menjawab dengan angka, berarti prompt Anda belum cukup kuat.
Perkuat kalimat izin mengaku tidak tahu, lalu jalankan ulang. Kegagalan di sini
adalah pelajaran yang lebih berguna daripada keberhasilan yang mulus.

In [6]:
_ = jawab("Berapa premi asuransi perjalanan ke luar negeri di LindungiKu?")

SUMBER YANG DIPAKAI
   - 04_prosedur_banding.md
   - 01_kebijakan_klaim_kesehatan.md
   - 02_kebijakan_klaim_kendaraan.md

JAWABAN
Kuring teu gaduh inpormasi ngeunaan premi asuransi perjalanan ka luar negeri di LindungiKu.


## Bagian 5 — Giliran Anda

Ajukan tiga pertanyaan Anda sendiri kepada sistem yang baru selesai dibangun.

1. Satu pertanyaan yang jawabannya jelas ada di dalam dokumen
2. Satu pertanyaan yang hanya dijawab separuh oleh dokumen
3. Satu pertanyaan yang sama sekali tidak dibahas dokumen

Dua pertanyaan pertama menunjukkan sistemnya bekerja. Pertanyaan ketiga
menunjukkan sistemnya jujur.

Ketika hasilnya terlihat aneh, jangan langsung menyalahkan modelnya. Periksa dulu
potongan mana yang terambil, memakai sel diagnosa di bawah. Hampir selalu
masalahnya ada di retrieval, bukan di generate.

In [10]:
PERTANYAAN_SAYA = [
    "apa saja perawatan yang tidak ditanggung oleh Lindungiku?",   # 1. jelas dijawab dokumen
    "Berapa maksimal peserta untuk produk Keluarga, dan berapa orang anak yang dijamin?",   # 2. dijawab sebagian
    "Kapan perusahaan Lindungiku berdiri ?",   # 3. tidak dibahas sama sekali
]

for p in PERTANYAAN_SAYA:
    if not p.strip():
        continue
    print("=" * 72)
    print(p)
    print("=" * 72)
    jawab(p)
    print()

apa saja perawatan yang tidak ditanggung oleh Lindungiku?
SUMBER YANG DIPAKAI
   - 01_kebijakan_klaim_kesehatan.md
   - 04_prosedur_banding.md
   - 02_kebijakan_klaim_kendaraan.md

JAWABAN
Perawatan anu teu ditanggung ku Lindungiku téh nyaéta:

1. **Perawatan kecantikan jeung estetika**  
2. **Perawatan gigi rutin di luar kecelakaan**  
3. **Program kesuburan jeung kehamilan dina dua taun kahiji polis**  
4. **Pengobatan alternatif tanpa rujukan dokter**  
5. **Perawatan akibat cedera anu disengaja**  
6. **Perawatan akibat kagiatan olahraga ekstrim anu teu dilaporkeun dina pendaftaran**

Berapa maksimal peserta untuk produk Keluarga, dan berapa orang anak yang dijamin?
SUMBER YANG DIPAKAI
   - 03_produk_dan_kode_polis.md
   - 03_produk_dan_kode_polis.md
   - 01_kebijakan_klaim_kesehatan.md

JAWABAN
Maksimal peserta pikeun produk Keluarga nyaéta **6**.  
Ngeunaan sabaraha anak anu dijamin, **henteu aya inpormasi** dina konteks nu disayogikeun.

Kapan perusahaan Lindungiku berdiri ?
SUM

### Sel diagnosa

Ketika jawabannya salah, jalankan ini dengan pertanyaan yang sama. Ia menunjukkan
apa yang sebenarnya dibaca model, sebelum dan sesudah re-ranking.

In [11]:
def diagnosa(pertanyaan, k_ambil=8):
    q = normalisasi(embed([pertanyaan], "RETRIEVAL_QUERY"))
    skor, baris = index.search(q, k_ambil)
    kandidat = [potongan[i] for i in baris[0]]
    nilai = reranker.predict([(pertanyaan, p["teks"]) for p in kandidat])

    print(f"{'#':<3} {'cosine':>7} {'rerank':>8}  sumber")
    print("-" * 60)
    for n, (s, r, p) in enumerate(zip(skor[0], nilai, kandidat), 1):
        print(f"{n:<3} {s:>7.3f} {r:>8.2f}  {p['sumber']}")
    print()
    print("Urutan berubah setelah re-ranking? Itu tandanya cross-encoder bekerja.")


diagnosa("Berapa lama masa tunggu klaim rawat inap?")

#    cosine   rerank  sumber
------------------------------------------------------------
1     0.748     4.16  01_kebijakan_klaim_kesehatan.md
2     0.741     6.40  01_kebijakan_klaim_kesehatan.md
3     0.735    -0.98  07_faq_internal_agen.md
4     0.728    -8.10  01_kebijakan_klaim_kesehatan.md
5     0.706    -1.21  07_faq_internal_agen.md
6     0.699    -9.34  01_kebijakan_klaim_kesehatan.md
7     0.697    -3.77  02_kebijakan_klaim_kendaraan.md
8     0.692    -8.18  02_kebijakan_klaim_kendaraan.md

Urutan berubah setelah re-ranking? Itu tandanya cross-encoder bekerja.
